# GP - SWIM Experiments

In [1]:
import torch
import gpytorch
import numpy as np
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

## STAGE 1: Create a TOY dataset and fit an Exact Gaussian Process 

In [ ]:
torch.manual_seed(42)

# ─── 1. Create dataset ───────────────────────────────────
N_train = 100
N_test  = 300

# Input: uniform in [-3, 3]
X_train = torch.linspace(-3, 3, N_train).unsqueeze(1)  # shape (100, 1)
y_train = torch.sin(X_train.squeeze()) + 0.1 * torch.randn(N_train)

X_test  = torch.linspace(-4, 4, N_test).unsqueeze(1)   # shape (300, 1)
y_test  = torch.sin(X_test.squeeze())                   # noiseless ground truth

print(f"X_train: {X_train.shape},  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape},   y_test:  {y_test.shape}")


X_train: torch.Size([100, 1]),  y_train: torch.Size([100])
X_test:  torch.Size([300, 1]),   y_test:  torch.Size([300])


In [3]:
# ─── 2. Define GP model ──────────────────────────────────
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, X_train, y_train, likelihood):
        super().__init__(X_train, y_train, likelihood)
        self.mean_module  = gpytorch.means.ConstantMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel()
        )

    def forward(self, x):
        mean  = self.mean_module(x)
        covar = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean, covar) # type: ignore

In [4]:
# ─── 3. Initialize ───────────────────────────────────────
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model      = ExactGPModel(X_train, y_train, likelihood)

In [5]:
# ─── 4. Train ────────────────────────────────────────────
model.train()
likelihood.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
mll       = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

num_iters = 100 
for i in range(num_iters):
    optimizer.zero_grad()
    loss = -mll(model(X_train), y_train) # type: ignore
    loss.backward()
    optimizer.step()

print(f"\nGP fitted successfully.")
print(f"  Length scale: {model.covar_module.base_kernel.lengthscale.item():.4f}")
print(f"  Output scale: {model.covar_module.outputscale.item():.4f}")
print(f"  Noise:        {likelihood.noise.item():.4f}")
print(f"  Mean const:   {model.mean_module.constant.item():.4f}") # type: ignore


GP fitted successfully.
  Length scale: 1.2596
  Output scale: 0.6832
  Noise:        0.0089
  Mean const:   0.1524


In [6]:
# ─── 5. Freeze GP ────────────────────────────────────────
model.eval()
likelihood.eval()
print(f"\nGP frozen. Ready for Stage 2 — pair sampling.")


GP frozen. Ready for Stage 2 — pair sampling.


## STAGE 2: GP Driven SWIM Scores

In [ ]:
def gp_swim_sample_pairs(
    X_train,          # torch tensor (N, d)
    y_train,          # torch tensor (N,)
    model,            # fitted frozen GP model
    likelihood,       # fitted frozen likelihood
    M,                # number of candidate pairs
    N_pairs,          # number of pairs to select (= layer_width equivalent)
    T=3,              # number of interior points per pair
    epsilon=1e-8,     # numerical stability
    random_seed=42
):
    rng = np.random.default_rng(random_seed)
    N = X_train.shape[0]

    # ── Step 1: Sample M candidate pairs ─────────────────
    # Same logic as SWIM — delta trick guarantees idx_from != idx_to
    idx_from = rng.integers(low=0, high=N, size=M)
    delta    = rng.integers(low=1, high=N-1, size=M)
    idx_to   = (idx_from + delta) % N

    x_a = X_train[idx_from]   # shape (M, d)
    x_b = X_train[idx_to]     # shape (M, d)
    y_a = y_train[idx_from]   # shape (M,)

    # ── Step 2: Create T interior points per pair ────────
    # t in {1/(T+1), 2/(T+1), ..., T/(T+1)} — avoids endpoints
    t_values = torch.linspace(0, 1, T+2)[1:-1]  # shape (T,)

    # x_t shape: (M, T, d)
    # x_a[:, None, :] broadcasts to (M, 1, d)
    x_interior = (
        x_a.unsqueeze(1) +
        t_values.view(1, T, 1) * (x_b - x_a).unsqueeze(1)
    )  # (M, T, d)

    # Flatten to (M*T, d) for single GP query
    x_interior_flat = x_interior.reshape(M * T, -1)

    # ── Step 3: Query frozen GP at interior points ───────
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        pred         = likelihood(model(x_interior_flat))
        mu_interior  = pred.mean.reshape(M, T)      # (M, T)
        std_interior = pred.variance.sqrt().reshape(M, T)  # (M, T)

    # ── Step 4: Compute GP-SWIM pair scores ──────────────
    # Numerator: mean variation from starting point y_a along segment
    # Use y_a directly (training point so mu*(x_a) ≈ y_a)
    variation   = (mu_interior - y_a.unsqueeze(1)).abs().mean(dim=1)  # (M,)

    # Denominator: mean uncertainty along segment
    uncertainty = std_interior.mean(dim=1) + epsilon                  # (M,)

    # GP-SWIM score = signal-to-noise ratio
    scores = variation / uncertainty   # (M,)

    # ── Step 5: Normalize to probabilities ───────────────
    scores_np = scores.numpy()

    if scores_np.sum() < epsilon:
        # Fallback to uniform if all scores collapse
        probabilities = np.ones(M) / M
    else:
        probabilities = scores_np / scores_np.sum()

    # ── Step 6: Sample N_pairs winners ───────────────────
    selected_idx = rng.choice(M, size=N_pairs, replace=True, p=probabilities)

    # Return selected pairs and their metadata
    return {
        'idx_from':      idx_from[selected_idx],       # original X_train indices
        'idx_to':        idx_to[selected_idx],
        'x_a':           x_a[selected_idx],            # (N_pairs, d)
        'x_b':           x_b[selected_idx],            # (N_pairs, d)
        'scores':        scores_np[selected_idx],      # (N_pairs,)
        'probabilities': probabilities,                 # (M,) full distribution
        'all_scores':    scores_np,                    # (M,) for inspection
    }